In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pydicom as dicom
import sys
import matplotlib.path as mplPath
import pandas as pd
import shutil
import os
import glob
import csv
from skimage import io

import torch
from torchvision import datasets
import torch.nn as nn
from torch.utils.data import Dataset
from torchvision.io import read_image
import torchvision.transforms as transforms
from torch.utils.data import Dataset, DataLoader,WeightedRandomSampler

In [ ]:
# defining transformation
train_transforms = transforms.Compose(
    [transforms.RandomHorizontalFlip(),
     transforms.RandomRotation(30),
     transforms.RandomVerticalFlip(),
     transforms.Resize((224, 224)),
    transforms.ToTensor()])

test_transforms = transforms.Compose(
    [transforms.Resize((224, 224)),
transforms.ToTensor()])

val_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor()
])

trainset = datasets.ImageFolder('Splitted2D/train', transform=train_transforms)
valset = datasets.ImageFolder('Splitted2D/val', transform = val_transforms)
testset = datasets.ImageFolder('Splitted2D/test', transform=test_transforms)

In [ ]:
idx2class_train = {v: k for k, v in trainset.class_to_idx.items()}
idex2class_val = {v: k for k, v in valset.class_to_idx.items()}
idex2class_test = {v: k for k, v in testset.class_to_idx.items()}

def get_class_distribution(dataset_obj, idx2class):
    count_dict = {k:0 for k,v in dataset_obj.class_to_idx.items()}
    
    for element in dataset_obj:
        y_lbl = element[1]
        y_lbl = idx2class[y_lbl]
        count_dict[y_lbl] += 1
            
    return count_dict
#print("Distribution of classes: \n", get_class_distribution(trainset))


In [ ]:
train_list = torch.tensor(trainset.targets)
class_count_train = [i for i in get_class_distribution(trainset, idx2class_train).values()]
train_weights = 1./torch.tensor(class_count_train, dtype=torch.float) 
train_weights_all = train_weights[train_list]

val_list = torch.tensor(valset.targets)
class_count_val = [i for i in get_class_distribution(valset, idx2class_train).values()]
val_weights = 1./torch.tensor(class_count_val, dtype = torch.float)
val_weights_all = val_weights[val_list]

test_list = torch.tensor(testset.targets)
class_count_test = [i for i in get_class_distribution(testset, idx2class_train).values()]
test_weights = 1./torch.tensor(class_count_test, dtype = torch.float)
test_weights_all = test_weights[test_list]

In [ ]:
for batch, (x, y) in enumerate(train_loader):
    print(len(x))

In [ ]:
train_weights_all

In [ ]:
weighted_train_sampler = WeightedRandomSampler(
    weights=train_weights_all,
    num_samples=len(train_weights_all),
    replacement=True
)
weighted_val_sampler = WeightedRandomSampler(
    weights = val_weights_all,
    num_samples = len(val_weights_all),
    replacement = True
)
weighted_test_sampler = WeightedRandomSampler(
    weights = test_weights_all,
    num_samples = len(test_weights_all),
    replacement = True
)

In [ ]:
batch_size = 16
train_loader = DataLoader(trainset, batch_size = batch_size, sampler = weighted_train_sampler)
val_loader = DataLoader(valset, batch_size = batch_size, sampler = weighted_val_sampler)
test_loader = DataLoader(testset, batch_size = batch_size, sampler = weighted_test_sampler)

In [ ]:
import monai

device = ("mps")
#model = monai.networks.nets.EfficientNetBN("efficientnet-b3", spatial_dims = 2, 
#                                          in_channels = 3, num_classes = 4, pretrained = True).to(device)
#model = monai.networks.nets.SEResNet50(spatial_dims = 2, in_channels = 3, num_classes = 4).to(device)
model = monai.networks.nets.resnet50(spatial_dims = 2, num_classes = 4).to(device)
model.train()
count = 0
# for child in model.children():
#     #print(count, child)
#     count += 1
#     if (count <= 5):
#         for param in child.parameters():
#             param.requires_grad = False

# model.last_linear = nn.Sequential(
#     nn.Linear(2048, 1024),
#     nn.ReLU(),
#     nn.Dropout(0.3, inplace = False),
#     nn.Linear(1024, 512),
#     nn.ReLU(),
#     nn.Dropout(0.3, inplace = False),
#     nn.Linear(256, 4),
#     nn.Softmax(dim = -1)
# )
#model.last_linear = nn.Linear(2048, 4)

In [ ]:
def train_loop(train_loader, val_loader, model, loss_fn, optimizer):
    size = len(train_loader.dataset)
    correct = 0
    samples = 0
    
    for batch, (X, y) in enumerate(train_loader):
        # Compute prediction and loss
        X, y = X.to(device), y.to(device)
        pred = model(X)
        loss = loss_fn(pred, y)
        #print(len(X))

        # Backpropagation
        optimizer.zero_grad()
#         for name, param in model.named_parameters():
#             if param.grad is not None:
#                 gradient_norm = torch.norm(param.grad)
#                 print(gradient_norm)
        loss.backward()
        optimizer.step()
        
        with torch.no_grad():
            correct += (pred.argmax(1) == y).type(torch.float).sum().item()
            samples += pred.size(0)
            if batch % 5 ==0:
                print(f"Acc: {float(correct) / float(samples) * 100:.2f}% Out of {samples} \n")
                correct = 0
                samples = 0

        if batch % 5 == 0:
            loss, current = loss.item(), (batch + 1) * len(X)
            print(f"loss: {loss:>7f}  [{current:>5d}/{size:>5d}]  \n")
    val_correct, val_samples = 0, 0
    for x, y in val_loader:
        x, y = x.to(device), y.to(device)
        pred = model(x)
        with torch.no_grad():
            val_correct += (pred.argmax(1) == y).type(torch.float).sum().item()
            val_samples += pred.size(0)
    print(f"validation accuracy: {val_correct / val_samples * 100:.2f}%")

def test_loop(dataloader, model, loss_fn):
    size = len(dataloader.dataset)
    num_batches = len(dataloader)
    test_loss, correct = 0, 0

    with torch.no_grad():
        for X, y in dataloader:
            X, y = X.to(device), y.to(device)
            pred = model(X)
            test_loss += loss_fn(pred, y).item()
            correct += (pred.argmax(1) == y).type(torch.float).sum().item()

    test_loss /= num_batches
    correct /= size
    print(f"Test Error: \n Accuracy: {(100*correct):>0.1f}% out of {size}, Avg loss: {test_loss:>8f} \n")

In [ ]:
criterion = nn.CrossEntropyLoss()
num_epochs = 40
optimizer = torch.optim.SGD(model.parameters(), lr= 0.003) 
for t in range(num_epochs):
    print(f"Epoch {t+1}\n-------------------------------")
    train_loop(train_loader, val_loader ,model, criterion, optimizer)

In [ ]:
for batch in val_loader:
    x, y = batch
    print(y)